# Tarea 2.2 — Preprocesamiento del Proyecto Telco Churn

**Curso:** MAI 540 — Módulo 2 · **Proyecto:** predicción de fuga de clientes (churn)

Este cuaderno construye la herramienta de preprocesamiento gobernada por `context.md` (Tarea 2.1). Cubre, en este orden:

1. Diagnóstico del estado del conjunto de datos y tratamiento de valores faltantes.
2. Duplicados y valores atípicos, con la evidencia detrás de cada decisión.
3. Codificación de variables categóricas y escalado de numéricas.
4. Selección de características.
5. El orden completo está diseñado para que **ningún estadístico usado por el pipeline se calcule con datos del conjunto de prueba** — la justificación de cada paso está en su celda de texto.

No se sube el CSV a este repositorio (restricción de `context.md`, sección "Datos personales"): la celda siguiente lo pide por el diálogo de carga de Colab.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

## Paso 0 — Carga de datos

Si este cuaderno corre en Google Colab, pide el archivo por el diálogo de carga (`files.upload()`). Si corre localmente, ajusta `DATA_PATH` a la ruta del CSV en tu equipo. En ningún caso el CSV se versiona en este repositorio.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    print("Sube el archivo 'datos Proyecto - Telco_Churn.csv' en el dialogo que se abre a continuacion...")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
else:
    DATA_PATH = "datos Proyecto - Telco_Churn.csv"  # ajusta la ruta si ejecutas localmente

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Paso 1 — Eliminar variables prohibidas (`context.md`)

`TotalCharges` (fuga de información), `customerID` (identificador directo) y `gender` (atributo protegido sin poder predictivo) se eliminan **antes de cualquier otra transformación**, tal como exige `context.md`.

In [ ]:
PROHIBIDAS = ["TotalCharges", "customerID", "gender"]
df = df.drop(columns=PROHIBIDAS)
print(f"Columnas tras eliminar prohibidas: {df.shape[1]}")

## Paso 2 — Recodificación de categorías redundantes (R2/R3 de `context.md`)

`"No internet service"` y `"No phone service"` ya están implícitas en `InternetService = No` y `PhoneService = No` respectivamente. Se recodifican a `"No"` para no duplicar la misma señal en varias columnas. Es una regla fija (no aprende nada de los datos), por lo que es segura de aplicar antes de particionar.

In [ ]:
SERVICIOS_INTERNET = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                       "TechSupport", "StreamingTV", "StreamingMovies"]
for c in SERVICIOS_INTERNET:
    df[c] = df[c].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")
print("Recodificado. Valores unicos de OnlineSecurity ahora:", df["OnlineSecurity"].unique())

## Paso 3 — Diagnóstico del estado del conjunto de datos

Reporte descriptivo (conteos, no estadísticos de un modelo) sobre el conjunto completo: nulos declarados, cadenas vacías, duplicados y valores atípicos. Como no calcula ni guarda ningún parámetro para el pipeline, es seguro hacerlo antes de particionar — es solo lectura.

In [ ]:
print("=== Nulos declarados (NaN) por columna ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else "Ninguno.")

print("\n=== Cadenas vacias por columna (texto) ===")
for c in df.select_dtypes(include="object").columns:
    vacios = (df[c].astype(str).str.strip() == "").sum()
    if vacios > 0:
        print(f"{c}: {vacios} vacios")
print("(TotalCharges ya no esta en el dataframe: se elimino en el Paso 1, junto con sus 7 valores vacios.)")

print("\n=== Duplicados (en las columnas que va a ver el modelo) ===")
dup_mask = df.duplicated(keep=False)
print("Filas involucradas en algun grupo de duplicados:", dup_mask.sum())
print("Grupos distintos de filas identicas:", df[dup_mask].drop_duplicates().shape[0])
print("Distribucion de tenure en esas filas:")
print(df.loc[dup_mask, "tenure"].value_counts().sort_index())

print("\n=== Atipicos (IQR, 1.5x) en variables numericas ===")
for c in ["tenure", "MonthlyCharges"]:
    s = df[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    print(f"{c}: rango=[{s.min()}, {s.max()}]  limites_IQR=[{lo:.2f}, {hi:.2f}]  atipicos={n_out}")

### Hallazgos y decisiones

- **Faltantes:** fuera de `TotalCharges` (ya excluida por fuga de información), **no hay ningún valor faltante** en las 17 columnas restantes: 0 `NaN` declarados y 0 cadenas vacías. No queda nada por imputar en el sentido estricto — ver la comparación de estrategias en el Paso 5, que se mantiene como medida defensiva del pipeline.
- **Duplicados:** en las 17 columnas que va a ver el modelo (tras quitar `TotalCharges`, `customerID` y `gender`), 42 filas (20 grupos) coinciden exactamente con al menos otra. El 81 % de esas filas (34 de 42) tiene `tenure = 1` (clientes en su primer mes, donde el catálogo de combinaciones de plan es pequeño: solo 316 combinaciones distintas de precio/contrato/internet/pago entre 388 clientes con `tenure = 1`, así que coincidencias exactas son estadísticamente esperables). El resto son un puñado de clientes de tenure más alto (2, 25 y 69 meses) que comparten el plan más barato posible (sin internet, solo teléfono) — la esquina de menor variedad del catálogo, donde volver a coincidir tampoco sorprende. Se investigaron antes de decidir (ver README, sección "Duplicados"): cada fila tiene un `customerID` original único y no son filas adyacentes en el archivo, lo que descarta un error de copiar-pegar. **Decisión: se conservan todas las filas** — coincidir en variables observables no implica ser el mismo registro capturado dos veces.
- **Atípicos:** el método IQR no encontró ningún valor atípico en `tenure` ni `MonthlyCharges`; los rangos observados (0–72 meses, USD 18.40–118.75) son consistentes con límites de negocio razonables. **Decisión: no se aplica ningún recorte (capping).**

## Paso 4 — Partición train/test (ANTES de ajustar cualquier transformación)

Regla no negociable de `context.md`: cualquier estadístico que el pipeline vaya a usar (mediana de imputación, categorías del one-hot, media/desviación del escalado, correlaciones de selección de variables) se calcula **solo con el conjunto de entrenamiento**, nunca con el de prueba — hacerlo al revés es una segunda forma de fuga de información.

In [ ]:
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"train: {X_train.shape}   test: {X_test.shape}")
print(f"Balance de Churn en train: {y_train.mean():.4f}   en test: {y_test.mean():.4f}")

## Paso 5 — Tratamiento de datos faltantes: comparación de estrategias

Aunque el diagnóstico del Paso 3 no encontró valores faltantes en las variables que sí entran al modelo, el pipeline incluye un imputador de todas formas, como medida defensiva ante datos futuros (un cliente nuevo con un campo vacío, por ejemplo). Se comparan dos estrategias para las numéricas, **ajustadas solo en train**:

In [ ]:
NUMERICAS = ["tenure", "MonthlyCharges"]

imp_mediana = SimpleImputer(strategy="median").fit(X_train[NUMERICAS])
imp_media = SimpleImputer(strategy="mean").fit(X_train[NUMERICAS])

print("Mediana (train):", dict(zip(NUMERICAS, imp_mediana.statistics_)))
print("Media (train):  ", dict(zip(NUMERICAS, imp_media.statistics_)))

**Decisión: mediana.** Con 0 valores faltantes actuales ambas estrategias producen el mismo resultado (no hay nada que imputar hoy), así que la elección no se puede justificar con una diferencia medible en este dataset. Se elige mediana por ser más robusta ante valores atípicos futuros que la media (una `MonthlyCharges` atípica en un lote de datos nuevo desplazaría la media pero no la mediana), consistente con el criterio de robustez que ya usa `context.md` para otras decisiones del proyecto.

Para las categóricas, el imputador defensivo usa `most_frequent` (moda): con variables binarias o de pocas categorías, es la única estrategia que no inventa una categoría inexistente (a diferencia de un valor constante como `"Desconocido"`, que además introduciría una categoría nueva que el one-hot ajustado en train no reconocería).

## Paso 6 — Codificación de variables categóricas y escalado

El algoritmo de referencia del curso hasta ahora es regresión logística (el mismo que usa `context.md` para sus cifras de referencia), así que el criterio de codificación se elige pensando en un modelo lineal:

- **Binarias (`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `MultipleLines` y las 6 de servicios de internet, ya reducidas a Sí/No en el Paso 2):** se mapean a 0/1 directamente. No hace falta one-hot para una variable de dos categorías — usarlo solo duplicaría la columna.
- **`SeniorCitizen`:** ya viene como 0/1 en el CSV original, no se toca.
- **`InternetService` y `PaymentMethod` (nominales, sin orden):** one-hot con `drop="first"` — evita la trampa de la variable dummy (colinealidad perfecta) en un modelo lineal.
- **`Contract` (con orden natural Month-to-month < One year < Two year):** se consideró codificación ordinal (0/1/2) por ese orden, pero **se descartó a favor de one-hot**. Motivo: un ordinal asume que el efecto de cada escalón es comparable, y los propios hallazgos de `context.md` muestran que no lo es — la tasa de churn cae de 42.8% (mensual) a 11.0% (anual), un salto de 31.8 puntos, y de 11.0% a 3.1% (bianual), un salto de solo 7.9 puntos. Un modelo lineal con `Contract` ordinal forzaría ese segundo salto a "valer" lo mismo que el primero; one-hot no impone esa restricción.
- **`tenure`, `MonthlyCharges` (numéricas continuas):** `StandardScaler`, ajustado solo en train. Necesario para regresión logística (la regularización y el optimizador basado en gradiente tratan todas las variables en la misma escala); no sería necesario para un modelo de árboles, pero tampoco los perjudica.

In [ ]:
BINARIAS = ["Partner", "Dependents", "PhoneService", "PaperlessBilling",
            "MultipleLines"] + SERVICIOS_INTERNET
NOMINALES = ["InternetService", "Contract", "PaymentMethod"]

def encode_binarias(d):
    out = d.copy()
    for c in BINARIAS:
        out[c] = (out[c] == "Yes").astype(int)
    return out

Xtr = encode_binarias(X_train)
Xte = encode_binarias(X_test)

ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
ohe.fit(Xtr[NOMINALES])  # fit SOLO en train
tr_ohe = pd.DataFrame(ohe.transform(Xtr[NOMINALES]), columns=ohe.get_feature_names_out(NOMINALES), index=Xtr.index)
te_ohe = pd.DataFrame(ohe.transform(Xte[NOMINALES]), columns=ohe.get_feature_names_out(NOMINALES), index=Xte.index)

scaler = StandardScaler().fit(Xtr[NUMERICAS])  # fit SOLO en train
tr_num = pd.DataFrame(scaler.transform(Xtr[NUMERICAS]), columns=NUMERICAS, index=Xtr.index)
te_num = pd.DataFrame(scaler.transform(Xte[NUMERICAS]), columns=NUMERICAS, index=Xte.index)

resto_train = Xtr.drop(columns=NOMINALES + NUMERICAS)
resto_test = Xte.drop(columns=NOMINALES + NUMERICAS)

train_full = pd.concat([tr_num, resto_train, tr_ohe], axis=1)
test_full = pd.concat([te_num, resto_test, te_ohe], axis=1)

print(f"train: {train_full.shape}   test: {test_full.shape}")
train_full.head()

## Paso 7 — Selección de características

Correlación de cada variable procesada con `Churn`, calculada **solo en train**, más conocimiento de dominio de `context.md`.

In [ ]:
corr_target = train_full.assign(Churn=y_train.values).corr(numeric_only=True)["Churn"].drop("Churn")
corr_target = corr_target.sort_values(key=abs, ascending=False)
print("=== Correlacion con Churn (train) ===")
print(corr_target.round(3).to_string())

corr_matrix = train_full.corr(numeric_only=True).abs()
np.fill_diagonal(corr_matrix.values, 0)
pares = corr_matrix.stack().reset_index()
pares.columns = ["var1", "var2", "corr"]
pares = pares[pares["var1"] < pares["var2"]].sort_values("corr", ascending=False)
print("\n=== Pares de variables mas correlacionados entre si (train) ===")
print(pares.head(10).to_string(index=False))

### Decisión: se conservan las 21 columnas procesadas

- **Ningún par de variables** supera 0.80 de correlación entre sí (el máximo es `InternetService_Fiber optic` vs. `MonthlyCharges` = 0.79), y ese par es explicable por dominio (la fibra es el servicio más caro) en vez de ser una redundancia pura — quitar una de las dos perdería información real (tipo de servicio ≠ precio).
- Variables con correlación débil frente a `Churn` (`PhoneService` = 0.034, `DeviceProtection` = -0.054, `MultipleLines` = 0.065) se revisaron una por una antes de decidir si se descartaban. En particular, `PhoneService` parecía redundante con `MultipleLines`, pero no lo es: tras la recodificación del Paso 2, `MultipleLines = No` mezcla dos poblaciones distintas (clientes sin servicio telefónico y clientes con una sola línea) que ya no se pueden separar sin `PhoneService`. Eliminarla perdería esa distinción.
- `SeniorCitizen`, `Partner` y `Dependents` se conservan como variables **condicionadas** (`context.md`): entran al conjunto procesado, pero `context.md` exige una auditoría de sesgo por subgrupos (diferencia de recall ≤ 0.10) antes de que cualquier modelo entrenado con ellas se declare válido. Esa auditoría es parte de la fase de modelado/evaluación del curso, no de esta tarea de preprocesamiento — queda pendiente y documentada aquí explícitamente, no ejecutada por adelantado.

No se aplicó ningún umbral automático de correlación para descartar variables: cada candidata débil se revisó con conocimiento de dominio antes de decidir, tal como pide el enunciado.